# VecteurPoynting — `vecteur_poynting`

`Postraitement_fields` · ASN 28 validation form (TRUST-LiMe).


## Introduction

Validation field of the **standalone magnetic (induction) equation** solved by
TRUST-LiMe:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

This notebook runs **one and only one test case**: ``vecteur_poynting`. Per the TRUST validation-form layout
(`Validation/Rapports_automatiques/<Category>/<FormName>/`), the other
`Postraitement_fields` cases live in their own sibling folders.

## What it validates

Ideal-MHD Poynting vector computed from the frozen velocity field
($E = -u \times b$). Vector case `nb_composantes 2` with uniform
$b = (1, 2)$, $u = (3, 4)$, $\mu_0 = 1$: $|b|^2 = 5$, $u \cdot b = 11$,
so $S = (4, -2)$ everywhere. Both components are checked at the probe.

## Contract checked

| # | Quantity | Formula |
|---|---|---|
| 1 | Poynting vector | $S = \mu_0^{-1}(|b|^2 u - (u \cdot b)\, b)$ |

Computed in the module by the `Champ_Fonc_MHD` post-treatment field
(`src/Champ_Fonc_MHD.{h,cpp}` + `src/MHD_posttraitement_kernel.h`).


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.


In [ ]:
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, sonde):
    """Return (time, values) for probe `sonde` of case `case` (dataset name)."""
    path = f"{BUILD}/{case}_{sonde}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def read_son_all(case, sonde):
    """Like read_son but returns every entry (components of a vector field)."""
    path = f"{BUILD}/{case}_{sonde}.son"
    son = SonFile(path, None)
    return [(e, np.asarray(son.getValues(e)[1], dtype=float)) for e in son.getEntries()]

def final_value(case, sonde):
    """Field value at the probe, at the last recorded time."""
    t, v = read_son(case, sonde)
    # the first line (t=0) is written before the problem is initialised
    return t, v[-1]

print("BUILD =", BUILD)


## Run

Runs on **2 processors** (`nbProcs=2` + `case.partition()`). The dataset's
commented `# BEGIN PARTITION` / `# BEGIN SCATTER` blocks let `trust -partition`
build the parallel `PAR_` variant (upstream `grad_rot.data` pattern).


In [ ]:
import pathlib, shutil

from trustutils import run

NB_PROCS = 2

run.reset()
case = run.addCase(".", "vecteur_poynting.data", nbProcs=NB_PROCS)
case.partition()          # trust -partition vecteur_poynting.data 2 -> PAR_vecteur_poynting.data + DOM_000*.Zones
run.printCases()
run.runCases()

# trustutils runs the case in its build tree; the .son probes are written next
# to the dataset in src/, so copy them where read_son() looks for them.
build = pathlib.Path(run.BUILD_DIRECTORY)
build.mkdir(parents=True, exist_ok=True)
for son in pathlib.Path("src").glob(f"{case.dataFileName_}_*.son"):
    shutil.copy2(son, build / son.name)


## Checks


In [ ]:
entries = read_son_all(case.dataFileName_, "SONDE_POYNTING")
for name, v in entries:
    print(name, "=", v[-1])
vals = {name: v[-1] for name, v in entries}

comps = [v[-1] for name, v in entries]
print(f"vecteur_poynting = ({comps[0]:.6g}, {comps[1]:.6g})")
assert abs(comps[0] - 4.0) < 1e-12 and abs(comps[1] + 2.0) < 1e-12, \
    f"vecteur_poynting wrong: {comps} (expected (4, -2))"
print("OK: vecteur_poynting = (|b|^2 u - (u.b) b) / mu0.")


## Computer Performance


In [ ]:
run.tablePerf()
